# Cell-state composition similarity between conditions

# 1. Imports and plotting style

In [1]:
import os
import warnings

import h5py
import anndata as ad
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

mpl.rcParams["pdf.fonttype"]       = 42
mpl.rcParams["ps.fonttype"]        = 42
sc.settings.set_figure_params(dpi=300, dpi_save=300)
mpl.rcParams["font.family"]        = "sans-serif"
mpl.rcParams["font.sans-serif"]    = ["Arial"]
mpl.rcParams["font.size"]          = 5
mpl.rcParams["font.weight"]        = "bold"
mpl.rcParams["axes.labelweight"]   = "bold"
mpl.rcParams["axes.titleweight"]   = "bold"
mpl.rcParams["figure.titleweight"] = "bold"
mpl.rcParams["axes.grid"]          = False
mpl.rcParams["axes.spines.top"]    = False
mpl.rcParams["axes.spines.right"]  = False
sns.set_style("ticks", {"axes.grid": False})

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

# 2. Configuration

In [2]:
# Input paths
GLOBAL_OBJ_PATH = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/Raw_UMI_merged_global_object_preclinical.h5ad"

# Output paths
RESULTS_DIR   = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pearson_states"
STATE_OUT_DIR = os.path.join(RESULTS_DIR, "cell_state_similarity_results")
os.makedirs(STATE_OUT_DIR, exist_ok=True)

# Metadata columns 
CONDITION_COL  = "Condition"
SAMPLE_COL     = "Sample_ID"
CELL_TYPE_COL  = "cell_type"
CELL_STATE_COL = "cell_state"
SURVIVAL_COL   = "Survival"     

# Conditions 
COND_ORDER = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]

# Cell states ==========================
EXCLUDE_CELL_TYPES = ["NC", "MC"]  # cell types left out of the cell-state analysis

HEATMAP_FIGSIZE      = (8, 6)      # (width, height) of the heatmap figure
HEATMAP_CMAP         = "RdBu_r"    # Colormap for the heatmap
HEATMAP_ANNOT_SIZE   = 9           # Font size for the numbers inside the heatmap
HEATMAP_LINEWIDTH    = 0.5         # Thickness of lines between heatmap cells
HEATMAP_TITLE_SIZE   = 5           # Font size for the heatmap title
DPI                  = 300 


# 3. Load the cell metadata and check the design

In [3]:
# 3. LOAD THE CELL METADATA AND CHECK THE DESIGN
# read the .obs table (one row per nucleus) of the global object to check no missing labels.

with h5py.File(GLOBAL_OBJ_PATH, "r") as f:
    obs = ad.io.read_elem(f["obs"])[[SAMPLE_COL, CONDITION_COL, CELL_TYPE_COL, CELL_STATE_COL, SURVIVAL_COL]]

missing = obs.isna().sum()
assert missing.sum() == 0, f"Missing labels:\n{missing[missing > 0]}"

conditions_per_heart = obs.groupby(SAMPLE_COL, observed=True)[CONDITION_COL].nunique()
assert (conditions_per_heart == 1).all(), f"Hearts in >1 condition: {list(conditions_per_heart[conditions_per_heart > 1].index)}"
assert (obs.groupby(SAMPLE_COL, observed=True)[SURVIVAL_COL].nunique() == 1).all(), "Hearts with >1 survival value"
assert set(COND_ORDER) == set(obs[CONDITION_COL].astype(str)), "Configured conditions differ from the object"

SAMPLE_CONDITION = obs.groupby(SAMPLE_COL, observed=True)[CONDITION_COL].first().astype(str)
SAMPLE_CONDITION.index = SAMPLE_CONDITION.index.astype(str)
SAMPLE_SURVIVAL = obs.groupby(SAMPLE_COL, observed=True)[SURVIVAL_COL].first().astype(str).astype(float)
SAMPLE_SURVIVAL.index = SAMPLE_SURVIVAL.index.astype(str)

design = pd.DataFrame({
    "hearts": SAMPLE_CONDITION.value_counts(),
    "nuclei": obs[CONDITION_COL].astype(str).value_counts(),
    "days_in_recipient": SAMPLE_SURVIVAL.groupby(SAMPLE_CONDITION).agg(lambda d: f"{d.min():g}-{d.max():g}"),
}).reindex(COND_ORDER)
print(f"{len(obs):,} nuclei from {SAMPLE_CONDITION.size} hearts\n")
print(design.to_string())

233,200 nuclei from 26 hearts

                        hearts  nuclei days_in_recipient
Condition                                               
Control                      4   29202               0-0
Prematurely-terminated       3   29136               0-2
3month-protocol              7   72986             51-90
6month-protocol              4   31989           120-195
PCMV/PRV                     3   20374             11-27
Rejection                    5   49513            18-196


# 4. Cell-state composition within each cell type

In [4]:
# cell_state compositions similariti per each cell type

def analyze_and_plot_cell_type(obs, cell_type, output_dir, cond_order):
    print(f"\n--- Processing cell type: {cell_type} ---")

    # 1. Nuclei of the current cell type
    subset_obs = obs[obs[CELL_TYPE_COL] == cell_type]
    if subset_obs.shape[0] == 0:
        print(f"No cells found for type '{cell_type}'. Skipping.")
        return None, None

    # 2. Cell state compositions (proportions)
    composition_data = {}
    conditions_present = subset_obs[CONDITION_COL].unique()
    cell_states = subset_obs[CELL_STATE_COL].unique()

    for cond in cond_order:
        if cond in conditions_present:
            cond_obs = subset_obs[subset_obs[CONDITION_COL] == cond]
            state_counts = cond_obs[CELL_STATE_COL].value_counts()

            # Normalize counts to get proportions within this cell type and condition
            state_proportions = state_counts / len(cond_obs)
            composition_data[cond] = state_proportions.reindex(cell_states, fill_value=0)
        else:
            composition_data[cond] = pd.Series(0.0, index=cell_states)

    df_composition = pd.DataFrame(composition_data)
    df_composition = df_composition.reindex(columns=cond_order, fill_value=0)

    # Prepare DataFrame for the Master proportions CSV
    df_for_master = df_composition.copy()
    df_for_master.index.name = CELL_STATE_COL
    df_for_master = df_for_master.reset_index()
    df_for_master.insert(0, 'Cell_Type', cell_type)

    df_corr_master = None

    # 3. Pearson correlation and heatmap
    if df_composition.shape[1] > 1:
        correlation_matrix = df_composition.corr(method='pearson')

        df_corr_master = correlation_matrix.copy()
        df_corr_master.index.name = "Condition_vs_Condition"
        df_corr_master = df_corr_master.reset_index()
        df_corr_master.insert(0, 'Cell_Type', cell_type)

        plt.figure(figsize=HEATMAP_FIGSIZE)
        sns.heatmap(
            correlation_matrix,
            annot=True,
            cmap=HEATMAP_CMAP,
            fmt=".2f",
            linewidths=HEATMAP_LINEWIDTH,
            annot_kws={"size": HEATMAP_ANNOT_SIZE, "weight": "bold"},
            cbar_kws={'label': 'Pearson Correlation Coefficient'}
        )

        plt.title(f'Pearson Correlation: {cell_type} Cell States',
                  fontsize=HEATMAP_TITLE_SIZE, fontweight="bold", pad=7)
        plt.xticks(rotation=90, ha='right', fontweight="bold")
        plt.yticks(rotation=0, fontweight="bold")
        plt.tight_layout()

        cell_type_clean = str(cell_type).replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
        pdf_filename = f"pearson_correlation_{cell_type_clean}.pdf"
        plt.savefig(os.path.join(output_dir, pdf_filename), dpi=DPI, format="pdf", bbox_inches="tight")
        print(f"Saved heatmap: {pdf_filename}")
        plt.close()
    else:
        print(f"Cannot compute correlation for {cell_type}: less than two conditions available.")

    return df_for_master, df_corr_master


all_cell_types = [ct for ct in obs[CELL_TYPE_COL].dropna().unique() if ct not in EXCLUDE_CELL_TYPES]

master_prop_list = []
master_corr_list = []
for ct in all_cell_types:
    df_prop, df_corr = analyze_and_plot_cell_type(obs, ct, STATE_OUT_DIR, COND_ORDER)
    if df_prop is not None:
        master_prop_list.append(df_prop)
    if df_corr is not None:
        master_corr_list.append(df_corr)

if master_prop_list:
    master_prop_path = os.path.join(STATE_OUT_DIR, "Master_CellState_Proportions_All_CellTypes.csv")
    pd.concat(master_prop_list, ignore_index=True).to_csv(master_prop_path, index=False)
    print(f"\nProportions CSV saved to: {master_prop_path}")
    if master_corr_list:
        master_corr_path = os.path.join(STATE_OUT_DIR, "Master_Pearson_Correlations_All_CellTypes.csv")
        pd.concat(master_corr_list, ignore_index=True).to_csv(master_corr_path, index=False)
        print(f"Correlations CSV saved to: {master_corr_path}")
else:
    print("\nNo data was processed. Master CSVs not created.")


--- Processing cell type: CM ---


Saved heatmap: pearson_correlation_CM.pdf

--- Processing cell type: FB ---


Saved heatmap: pearson_correlation_FB.pdf

--- Processing cell type: EC ---


Saved heatmap: pearson_correlation_EC.pdf

--- Processing cell type: Mye_sus ---


Saved heatmap: pearson_correlation_Mye_sus.pdf

--- Processing cell type: Lymph_sus ---


Saved heatmap: pearson_correlation_Lymph_sus.pdf

--- Processing cell type: Lymph_papio ---


Saved heatmap: pearson_correlation_Lymph_papio.pdf

--- Processing cell type: Mye_papio ---


Saved heatmap: pearson_correlation_Mye_papio.pdf

--- Processing cell type: EC_papio ---


Saved heatmap: pearson_correlation_EC_papio.pdf

Proportions CSV saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pearson_states/cell_state_similarity_results/Master_CellState_Proportions_All_CellTypes.csv
Correlations CSV saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/pearson_states/cell_state_similarity_results/Master_Pearson_Correlations_All_CellTypes.csv
